# 05 — v2 Results: HST-only Sources

This notebook explores the output of `bp3m-v2`, which extends the astrometric
catalogue to include HST-detected sources with no Gaia counterpart.

**Requires:** a completed `bp3m-v2` run (i.e. `BP3M_v2_results/` and `hst_xmatch/` must exist).

**Plots:** Gaia-matched vs HST-only populations · VPD · PM uncertainty comparison · CMD

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = '..'
FIELD_NAME = 'Sculptor_dSph'
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

from bp3m.pipeline.explore_utils import (
    load_gaia_catalog, load_bp3m_results,
    gaia_pm_sigma, bp3m_pm_sigma, bp3m_full_cov, vpd_limits,
)

DATA = Path(OUTPUT_DIR)
field_dir = DATA / FIELD_NAME

# Load v1 BP3M results
bp3m_v1 = load_bp3m_results(field_dir / 'BP3M_results')
stars_v1 = bp3m_v1['stars']

# Load v2 BP3M results
try:
    bp3m_v2 = load_bp3m_results(field_dir / 'BP3M_v2_results')
    stars_v2 = bp3m_v2['stars']
    print(f'v2 stars: {len(stars_v2)}')
except FileNotFoundError:
    stars_v2 = None
    print('BP3M_v2_results not found — run bp3m-v2 first.')

# Load master HST cross-match catalog
master_csv = field_dir / 'hst_xmatch' / 'master_combined_v2.csv'
try:
    master = pd.read_csv(master_csv, low_memory=False)
    print(f'Master catalog: {len(master)} sources')
except FileNotFoundError:
    master = None
    print(f'master_combined_v2.csv not found at {master_csv}')

# Load Gaia catalog
_gaia_files = sorted((field_dir / 'Gaia').glob('*_gaia.csv'))
gaia = load_gaia_catalog(_gaia_files[0])

# 2p solution masks — stars with no Gaia PM/parallax measurement
# Apply to any Gaia-derived PM or parallax uncertainty before plotting
_gaia_2p      = ~np.isfinite(gaia['pmra'].values) | ~np.isfinite(gaia['pmdec'].values)
_gaia_2p_v1   = ~np.isfinite(stars_v1['pmra'].values) | ~np.isfinite(stars_v1['pmdec'].values)

print(f'v1 stars: {len(stars_v1)}   Gaia: {len(gaia)}')

In [ ]:
# Separate Gaia-matched and HST-only sources in v2 results
# HST-only sources have negative Gaia_id (synthetic IDs assigned by bp3m)
if stars_v2 is not None:
    from bp3m.pipeline.catalog_utils import pm_uncertainty

    gaia_ids_v2 = stars_v2['Gaia_id'].values.astype(np.int64)
    is_gaia_matched = gaia_ids_v2 > 0
    is_hst_only     = gaia_ids_v2 < 0

    print(f'v2 Gaia-matched: {is_gaia_matched.sum()}')
    print(f'v2 HST-only:     {is_hst_only.sum()}')

    # Quality mask for HST-only stars: exclude poor PM solutions
    C_full_v2     = bp3m_full_cov(bp3m_v2)
    sig_pm_v2_all = pm_uncertainty(C_full_v2)   # geometric mean, mas/yr

    pm_size_v2 = np.sqrt(
        stars_v2['pmra_bp3m'].values**2 + stars_v2['pmdec_bp3m'].values**2
    )

    good_v2 = (
        np.where(np.isfinite(sig_pm_v2_all), sig_pm_v2_all, np.inf) <= 5
    ) & (
        np.where(np.isfinite(pm_size_v2), pm_size_v2, np.inf) <= 100
    )
    # Always keep Gaia-matched stars (already quality-controlled by Gaia catalog)
    good_v2 |= is_gaia_matched

    n_before = is_hst_only.sum()
    n_after  = (is_hst_only & good_v2).sum()
    print(f'HST-only after quality cut (σ_PM≤5, |PM|≤100): {n_after} / {n_before}')

    pmra_v2_gaia  = stars_v2.loc[is_gaia_matched, 'pmra_bp3m'].values
    pmdec_v2_gaia = stars_v2.loc[is_gaia_matched, 'pmdec_bp3m'].values
    pmra_v2_hst   = stars_v2.loc[is_hst_only & good_v2, 'pmra_bp3m'].values
    pmdec_v2_hst  = stars_v2.loc[is_hst_only & good_v2, 'pmdec_bp3m'].values
    gmag_v2_gaia  = stars_v2.loc[is_gaia_matched, 'gmag'].values
    n_hst_v2      = stars_v2['n_hst_used'].values

In [ ]:
# ── VPD: v2 Gaia-matched vs HST-only ─────────────────────────────────────────
if stars_v2 is not None:
    xlim, ylim = vpd_limits(
        np.concatenate([pmra_v2_gaia, pmra_v2_hst]),
        np.concatenate([pmdec_v2_gaia, pmdec_v2_hst]),
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

    axes[0].scatter(pmra_v2_gaia, pmdec_v2_gaia,
                    s=4, alpha=0.5, c='steelblue', rasterized=True,
                    label=f'Gaia-matched ({is_gaia_matched.sum()})')
    axes[0].set_title('v2 — Gaia-matched sources')

    axes[1].scatter(pmra_v2_hst, pmdec_v2_hst,
                    s=4, alpha=0.5, c='darkorange', rasterized=True,
                    label=f'HST-only ({is_hst_only.sum()})')
    axes[1].set_title('v2 — HST-only sources')

    for ax in axes:
        ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.3)
        ax.axvline(0, color='k', lw=0.5, ls='--', alpha=0.3)
        ax.set_xlim(*xlim); ax.set_ylim(*ylim)
        ax.set_xlabel(r'$\mu_{\alpha^*}$ (mas yr$^{-1}$)')
        ax.legend(fontsize=9)
    axes[0].set_ylabel(r'$\mu_\delta$ (mas yr$^{-1}$)')

    fig.suptitle(f'{FIELD_NAME} — v2 Vector Point Diagram', fontsize=13)
    plt.tight_layout()
    plt.savefig(field_dir / 'plots_05_vpd_v2.png', dpi=150)
    plt.show()

In [ ]:
# ── PM uncertainty comparison: Gaia / v1 / v2 ────────────────────────────────
if stars_v2 is not None:
    sig_pm_gaia = gaia_pm_sigma(gaia)
    sig_pm_v1   = bp3m_pm_sigma(bp3m_v1)

    # Mask Gaia 2p stars — no PM measurement, uncertainty is meaningless
    sig_pm_gaia = np.where(_gaia_2p,    np.nan, sig_pm_gaia)
    sig_pm_v1   = np.where(_gaia_2p_v1, np.nan, sig_pm_v1)

    sig_pm_v2_gaia = sig_pm_v2_all[is_gaia_matched]
    sig_pm_v2_hst  = sig_pm_v2_all[is_hst_only & good_v2]
    gmag_gaia_all  = gaia['gmag'].values

    # ── Gaia-matched comparison (vs Gaia G magnitude) ─────────────────────────
    fig, ax = plt.subplots(figsize=(9, 6))

    ok_g = np.isfinite(gmag_gaia_all) & np.isfinite(sig_pm_gaia)
    ax.scatter(gmag_gaia_all[ok_g], sig_pm_gaia[ok_g],
               s=4, alpha=0.3, c='grey', rasterized=True, label='Gaia DR3')

    ok_v1 = np.isfinite(stars_v1['gmag'].values) & np.isfinite(sig_pm_v1)
    ax.scatter(stars_v1['gmag'].values[ok_v1], sig_pm_v1[ok_v1],
               s=4, alpha=0.4, c='steelblue', rasterized=True, label='BP3M v1')

    ok_v2g = np.isfinite(gmag_v2_gaia) & np.isfinite(sig_pm_v2_gaia)
    ax.scatter(gmag_v2_gaia[ok_v2g], sig_pm_v2_gaia[ok_v2g],
               s=4, alpha=0.5, c='darkorange', rasterized=True,
               label='BP3M v2 (Gaia-matched)')

    ax.set_xlabel('Gaia G (mag)')
    ax.set_ylabel(r'$(\det C_\mu)^{1/4}$ (mas yr$^{-1}$)')
    ax.set_yscale('log')
    ax.legend(fontsize=9)
    ax.set_title(f'{FIELD_NAME} — PM uncertainty: Gaia / v1 / v2 Gaia-matched')
    fig.tight_layout()
    fig.savefig(field_dir / 'plots_05_pm_unc_comparison.png', dpi=150)
    plt.show()

    # ── HST-only: PM uncertainty vs N detections ──────────────────────────────
    ok_v2h = np.isfinite(sig_pm_v2_hst)
    n_hst_v2_hst = n_hst_v2[is_hst_only & good_v2]

    fig2, ax2 = plt.subplots(figsize=(7, 5))
    ax2.scatter(n_hst_v2_hst[ok_v2h], sig_pm_v2_hst[ok_v2h],
                s=6, alpha=0.5, c='darkorange', rasterized=True)
    ax2.set_xlabel('N HST detections')
    ax2.set_ylabel(r'$\sigma_\mu$ (mas yr$^{-1}$)')
    ax2.set_yscale('log')
    ax2.set_title(f'{FIELD_NAME} — v2 HST-only PM uncertainty vs N detections')
    plt.tight_layout()
    plt.savefig(field_dir / 'plots_05_pm_unc_hst_only.png', dpi=150)
    plt.show()

In [ ]:
# ── PM uncertainty vs HST filter magnitude ───────────────────────────────────
# Shows how BP3M v2 reaches much fainter than Gaia alone in each filter.
if master is not None and stars_v2 is not None:
    # Quality filter applied directly to master using its own sigma columns
    sig_pm_master = np.sqrt(np.abs(
        master['sigma_pmra'].values * master['sigma_pmdec'].values
    ))
    pm_size_master = master['pm_size_masyr'].values if 'pm_size_masyr' in master.columns \
                     else np.zeros(len(master))
    quality_master = (
        np.where(np.isfinite(sig_pm_master),  sig_pm_master,  np.inf) <= 5
    ) & (
        np.where(np.isfinite(pm_size_master), pm_size_master, np.inf) <= 100
    )
    master_good = master[quality_master].reset_index(drop=True)
    sig_pm_good  = sig_pm_master[quality_master]

    hst_mag_cols = sorted(c for c in master_good.columns if c.startswith('mag_wmean_'))
    filters = [c.replace('mag_wmean_', '') for c in hst_mag_cols]

    if not hst_mag_cols:
        print('No mag_wmean_* columns in master catalog.')
    else:
        cmap = plt.get_cmap('tab10', len(filters))
        fig, ax = plt.subplots(figsize=(10, 6))

        # Gaia DR3 comparison (2p-masked)
        sig_pm_gaia_plot = gaia_pm_sigma(gaia)
        sig_pm_gaia_plot = np.where(_gaia_2p, np.nan, sig_pm_gaia_plot)
        ok_g = np.isfinite(gaia['gmag'].values) & np.isfinite(sig_pm_gaia_plot)
        ax.scatter(gaia['gmag'].values[ok_g], sig_pm_gaia_plot[ok_g],
                   s=3, alpha=0.2, c='grey', rasterized=True, label='Gaia DR3')

        # BP3M v2 per HST filter
        for i, (col, filt) in enumerate(zip(hst_mag_cols, filters)):
            mag = master_good[col].values
            ok  = np.isfinite(mag) & np.isfinite(sig_pm_good) & (sig_pm_good > 0)
            ax.scatter(mag[ok], sig_pm_good[ok], s=4, alpha=0.4,
                       color=cmap(i), rasterized=True, label=f'BP3M v2 {filt}')

        ax.set_xlabel('Magnitude (mag)')
        ax.set_ylabel(r'$(\det C_\mu)^{1/4}$ (mas yr$^{-1}$)')
        ax.set_yscale('log')
        ax.legend(fontsize=9, loc='upper left')
        ax.set_title(f'{FIELD_NAME} — PM uncertainty vs HST filter magnitude')
        plt.tight_layout()
        plt.savefig(field_dir / 'plots_05_pm_unc_vs_hst_mag.png', dpi=150)
        plt.show()

In [ ]:
# ── All pairwise CMDs (replicates bp3m-v2 pipeline output) ───────────────────
# Uses the same _plot_cmds function that the pipeline writes to cmds_v2*.png.
# The Gaia CMD panels inside _plot_cmds are restricted to HST-matched stars
# by passing only the matched subset of the Gaia catalog.
if master is not None:
    from bp3m.pipeline.hst_catalog_crossmatch import _plot_cmds

    # Restrict Gaia to only stars that have an HST cross-match
    matched_gaia_ids = set(master['gaia_source_id'].dropna().astype(np.int64).unique())
    gaia_matched = gaia[gaia['source_id'].isin(matched_gaia_ids)].reset_index(drop=True)

    pm_size = master['pm_size_masyr'].values if 'pm_size_masyr' in master.columns else None

    _plot_cmds(
        master, gaia_matched,
        output_path=field_dir / 'plots_05_cmds_v2.png',
        title=f'{FIELD_NAME} — v2 pairwise CMDs',
        pm_size=pm_size,
    )
    from IPython.display import Image, display
    display(Image(str(field_dir / 'plots_05_cmds_v2.png')))
else:
    print('master_combined_v2.csv required for CMD plots.')

In [ ]:
# ── Sky distribution: v2 Gaia-matched vs HST-only ────────────────────────────
if stars_v2 is not None and 'ra' in stars_v2.columns and 'dec' in stars_v2.columns:
    ra_gaia  = stars_v2.loc[is_gaia_matched, 'ra'].values
    dec_gaia = stars_v2.loc[is_gaia_matched, 'dec'].values
    ra_hst   = stars_v2.loc[is_hst_only & good_v2, 'ra'].values
    dec_hst  = stars_v2.loc[is_hst_only & good_v2, 'dec'].values

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(ra_gaia, dec_gaia, s=3, alpha=0.4, c='steelblue',
               rasterized=True, label=f'Gaia-matched ({is_gaia_matched.sum()})')
    ax.scatter(ra_hst, dec_hst, s=3, alpha=0.4, c='darkorange',
               rasterized=True, label=f'HST-only ({len(ra_hst)})')

    all_ra  = np.concatenate([ra_gaia, ra_hst])
    all_dec = np.concatenate([dec_gaia, dec_hst])
    pad_ra  = (all_ra.max()  - all_ra.min())  * 0.05
    pad_dec = (all_dec.max() - all_dec.min()) * 0.05
    ax.set_xlim(all_ra.max() + pad_ra, all_ra.min() - pad_ra)
    ax.set_ylim(all_dec.min() - pad_dec, all_dec.max() + pad_dec)
    ax.set_xlabel('R.A. (deg)')
    ax.set_ylabel('Dec. (deg)')
    ax.set_title(f'{FIELD_NAME} — v2 sky distribution')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(field_dir / 'plots_05_sky_v2.png', dpi=150)
    plt.show()